In [21]:
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import OneHotEncoder,StandardScaler,PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV,RandomizedSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.metrics import r2_score, root_mean_squared_error
from scipy.stats import randint


In [22]:
df =  pd.read_csv("Student_DataSet.csv")
y = df["math_percentage"]
x =df.drop(columns=["math_percentage"])
encode_colums=["sex","race/ethnicity","parental_level_of_education","lunch","test_preparation_course"]
x

,sex,race/ethnicity,parental_level_of_education,lunch,test_preparation_course,reading_percentage,writing_percentage
0,F,group B,bachelor's degree,standard,none,0.72,0.74
1,F,group C,some college,standard,completed,0.90,0.88
2,F,group B,master's degree,standard,none,0.95,0.93
3,M,group A,associate's degree,free/reduced,none,0.57,0.44
4,M,group C,some college,standard,none,0.78,0.75
...,...,...,...,...,...,...,...
995,F,group E,master's degree,standard,completed,0.99,0.95
996,M,group C,high school,free/reduced,none,0.55,0.55
997,F,group C,high school,free/reduced,completed,0.71,0.65
998,F,group D,some college,standard,completed,0.78,0.77


In [23]:
ct = ColumnTransformer(transformers=[("encoder",OneHotEncoder(),encode_colums)],remainder="passthrough")
x = ct.fit_transform(x)

In [24]:
x_train,x_test,y_train, y_test =train_test_split(x,y, test_size=0.2,random_state=0)

In [25]:
def report(models,y_test):
        for name, pred in models:
            print(f"{name} Report\nR2 score:{r2_score(y_test,pred)}\nRMSE:{root_mean_squared_error(y_test,pred)}") 
    

In [26]:
lin_reg = LinearRegression()
lin_reg.fit(x_train,y_train)
lin_reg_pred = lin_reg.predict(x_test)


In [27]:

poly = PolynomialFeatures(degree=2)
x_poly = poly.fit_transform(x_train)
poly_reg = LinearRegression()
poly_reg.fit(x_poly,y_train)
poly_reg_pred = poly_reg.predict(poly.transform(x_test))

In [28]:
svm = SVR(kernel="rbf")
svm.fit(x_train,y_train)
svm_pred = svm.predict(x_test)


# Decision Tree

In [29]:
dt = DecisionTreeRegressor()
dt_grid= {
    "max_depth":[i for i in range(1,10)]
}
grid=GridSearchCV(estimator=dt,param_grid=dt_grid,cv=3)
grid.fit(x_train,y_train)

dt_pred = grid.predict(x_test)


# Random Forest

In [30]:
rf = RandomForestRegressor(n_estimators=150)
rf_grid= {
    "n_estimators" : [i for i in range(0,500,100)],
    "max_depth" : [j for j in range(1,10)]
}
rf_cv = RandomizedSearchCV(rf,rf_grid,n_iter=5,cv=5)
rf_cv.fit(x_train,y_train)
rf_pred = rf_cv.predict(x_test)

c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
5 fits failed out of a total of 25.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py", line 1358, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\U

In [31]:
from sklearn.linear_model import Ridge

# Ridge handles correlated features better than simple LinearRegression
ridge = Ridge(alpha=1.0)
ridge.fit(x_train, y_train)
ridge_pred = ridge.predict(x_test)

In [32]:
from sklearn.model_selection import GridSearchCV
# Assuming gb is the base estimator; if not, define it
gb = GradientBoostingRegressor()  
gb_grid = {
    "n_estimators": [a for a in range(100, 500, 50)],
    "learning_rate": [0.05 + i * 0.2 for i in range(2)],  # [0.05, 0.25]
    "max_depth": [3, 5, 6]
}

model = GridSearchCV(gb, gb_grid, cv=5) 
model.fit(x_train, y_train)
gb_pred = model.predict(x_test)

In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor(random_state=42)

xgb_grid = {
    "n_estimators": [100, 200, 300, 400, 500],
    "learning_rate": [0.01, 0.05, 0.1, 0.2, 0.3],
    "max_depth": [3, 5, 6, 7],
    "subsample": [0.7, 0.8, 1.0],          # rows sampled per tree
    "colsample_bytree": [0.7, 0.8, 1.0]    # features sampled per tree
}

model = GridSearchCV(xgb, xgb_grid, cv=5, scoring="r2", n_jobs=-1)
model.fit(x_train, y_train)
xgb_pred=model.predict(x_test)

print(model.best_params_)
print(model.best_score_)

{'colsample_bytree': 0.7, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.8}
0.8514152014078258


In [36]:
xgb_pred=model.predict(x_test)

In [37]:
predictions =[("Linear Regression",lin_reg_pred),("Polynomial Regression",poly_reg_pred),("SVM",svm_pred),("Decision Tree", dt_pred),("Random Forest",rf_pred),("Ridge",ridge_pred),("XGBoost",xgb_pred ),("GradientBoosting",gb_pred)]
report(predictions,y_test)

Linear Regression Report
R2 score:0.8629895363812153
RMSE:0.055719646690156235
Polynomial Regression Report
R2 score:0.85777863657398
RMSE:0.056769347808934355
SVM Report
R2 score:0.8396048881919675
RMSE:0.06028746882054266
Decision Tree Report
R2 score:0.8338114153152061
RMSE:0.061366602636376895
Random Forest Report
R2 score:0.8615361238327236
RMSE:0.05601440518447616
Ridge Report
R2 score:0.8661475859946599
RMSE:0.0550737425434957
XGBoost Report
R2 score:0.8678071682481202
RMSE:0.054731259060528434
GradientBoosting Report
R2 score:0.8635120730826796
RMSE:0.05561329213429812


In [40]:
from sklearn.model_selection import GridSearchCV

xgb = XGBRegressor(random_state=42)

param_grid_1 = {
    "n_estimators": [100, 200, 300, 400, 500],
    "learning_rate": [0.01, 0.05, 0.1, 0.2, 0.3]
}

grid_1 = GridSearchCV(xgb, param_grid_1, cv=5, scoring="r2", n_jobs=-1, verbose=1)
grid_1.fit(x_train, y_train)

print("Best Params:", grid_1.best_params_)
print("Best R2:", grid_1.best_score_)

Fitting 5 folds for each of 25 candidates, totalling 125 fits
Best Params: {'learning_rate': 0.01, 'n_estimators': 400}
Best R2: 0.8271484202727837


In [42]:
xgb = XGBRegressor(random_state=42, learning_rate=0.01, n_estimators=400, max_depth=4)

param_grid_3 = {
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0]
}

grid_3 = GridSearchCV(xgb, param_grid_3, cv=5, scoring="r2", n_jobs=-1, verbose=1)
grid_3.fit(x_train, y_train)

print("Best Params:", grid_3.best_params_)
print("Best R2:", grid_3.best_score_)

Fitting 5 folds for each of 25 candidates, totalling 125 fits
Best Params: {'colsample_bytree': 0.9, 'subsample': 0.6}
Best R2: 0.8475212848634918
